# Notebook khởi đầu

1. Bắt buộc dùng **link tài nguyên chỉ đọc của BTC** ở mục **Tài nguyên → Bộ dữ liệu**. Dán link vào `DATASET_URL` bên dưới; notebook sẽ tải và đọc tệp Test từ chính link đó (link S3, Google Drive hay máy chủ riêng đều được).
2. Điền tên cột, `TEST_FILE` theo cuộc thi; tự hoàn thiện hai bước `TODO` (tiền xử lý, huấn luyện/dự đoán).
3. Chạy lại notebook để tạo `prediction.csv` rồi nộp CSV cùng notebook `.ipynb`.

In [ ]:
# 1. THƯ VIỆN VÀ SEED
import random
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Chạy một lần trước khi tải dữ liệu nếu môi trường chưa có gdown.
%pip install -q gdown
import gdown

In [ ]:
# 2. CẤU HÌNH — điền theo cuộc thi đang tham gia.
DATASET_URL = ""  # Bắt buộc: dán link "Bộ dữ liệu" của BTC (link tài nguyên chỉ đọc).
ID_COLUMN = "id"
PREDICTION_COLUMNS = ["prediction"]  # Tất cả cột dự đoán theo đúng thứ tự trong Hướng dẫn.
TEST_FILE = "test.csv"  # Tên tệp Test trong bộ dữ liệu BTC.
OUTPUT_FILE = "prediction.csv"

In [ ]:
# 3. TẢI VÀ ĐỌC DỮ LIỆU TỪ LINK TÀI NGUYÊN CỦA BTC
from urllib.parse import parse_qs, urlparse

assert DATASET_URL.startswith("https://"), "Dán link tài nguyên BTC (https://) vào DATASET_URL."
url = urlparse(DATASET_URL)
host = (url.hostname or "").lower()
DATA_DIR = Path("dataset_btc")
if host in {"drive.google.com", "drive.usercontent.google.com"} and "/folders/" in url.path:
    # Google Drive: thư mục BTC — gdown tải cả thư mục.
    downloaded = gdown.download_folder(DATASET_URL, output=str(DATA_DIR), quiet=False)
elif host in {"drive.google.com", "drive.usercontent.google.com"} and (
    ("/file/" in url.path and "/d/" in url.path)
    or (url.path in ("/open", "/uc", "/download") and parse_qs(url.query).get("id"))
):
    # Google Drive: một tệp BTC — gdown tải về đúng tên TEST_FILE.
    DATA_DIR.mkdir(exist_ok=True)
    downloaded = [gdown.download(DATASET_URL, output=str(DATA_DIR / TEST_FILE), quiet=False)]
else:
    # Link ngoài Google Drive (S3, máy chủ khác...): tải trực tiếp; nếu là tệp .zip thì giải nén.
    DATA_DIR.mkdir(exist_ok=True)
    archive = DATA_DIR / (Path(url.path).name or "dataset")
    urlretrieve(DATASET_URL, str(archive))
    if archive.suffix.lower() == ".zip":
        with zipfile.ZipFile(archive) as zip_file:
            zip_file.extractall(DATA_DIR)
            downloaded = [
                str(DATA_DIR / member.filename)
                for member in zip_file.infolist()
                if not member.is_dir()
            ]
        archive.unlink()
    else:
        downloaded = [str(archive)]

assert downloaded and all(downloaded), "Không tải được dữ liệu từ link tài nguyên BTC."
test_files = [Path(path) for path in downloaded if Path(path).name == TEST_FILE]
assert len(test_files) == 1, f"Bộ dữ liệu BTC cần đúng một tệp {TEST_FILE}."
test_path = test_files[0]
test = pd.read_csv(test_path)
print("Đã đọc:", test_path, "| Kích thước Test:", test.shape)
test.head()

In [ ]:
# 4. KIỂM TRA CỘT ID
assert ID_COLUMN in test.columns, f"Thiếu cột {ID_COLUMN!r} trong {test_path}."
ids = test[ID_COLUMN]
assert ids.notna().all(), "Cột ID có giá trị rỗng."
assert not ids.duplicated().any(), "Cột ID bị trùng."


In [ ]:
# 5. TIỀN XỬ LÝ DỮ LIỆU — TODO: đọc dữ liệu train từ DATA_DIR và xử lý đặc trưng.


In [ ]:
# 6. HUẤN LUYỆN MÔ HÌNH — TODO: tạo predictions cho từng dòng test.
# predictions: DataFrame có các cột PREDICTION_COLUMNS theo đúng thứ tự, cùng số dòng với test.


In [ ]:
# 7. GHÉP DỰ ĐOÁN VỚI ID (chạy sau khi đã hoàn thiện TODO)
assert PREDICTION_COLUMNS and ID_COLUMN not in PREDICTION_COLUMNS
assert list(predictions.columns) == PREDICTION_COLUMNS
assert len(predictions) == len(test)
submission = pd.concat([test[[ID_COLUMN]].reset_index(drop=True), predictions.reset_index(drop=True)], axis=1)


In [ ]:
# 8. XUẤT CSV — đúng tên và thứ tự cột cuộc thi yêu cầu.
assert list(submission.columns) == [ID_COLUMN, *PREDICTION_COLUMNS]
assert len(submission) == len(test)
submission.to_csv(OUTPUT_FILE, index=False)
written = pd.read_csv(OUTPUT_FILE)
assert list(written.columns) == [ID_COLUMN, *PREDICTION_COLUMNS]
assert len(written) == len(test)
print(f"Đã ghi {OUTPUT_FILE}: {len(written)} dòng.")
